# 02 - Forecasting Experiments

Compare ARIMA, Prophet, seasonal-naive and the ensemble with the same rolling-origin backtest the Forecast Agent uses, then check against the real hold-out week.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.width", 160)

In [ ]:
from src.config.settings import load_settings
from src.data.cleaning import clean_item_sales
from src.data.features import (PROPHET_REGRESSORS, build_features, build_future_frame,
                               prophet_holidays)
from src.data.repository import DataRepository
from src.evaluation.backtest import ensemble_backtest, rolling_origin_backtest
from src.evaluation.metrics import compute_metrics
from src.models import ARIMAForecaster, ProphetForecaster, SeasonalNaiveForecaster

settings = load_settings(load_env=False)
repo = DataRepository(settings)
ITEM = "FOODS_3_064"
clean, quality = clean_item_sales(repo.item_sales(ITEM))
feats = build_features(clean)
future = build_future_frame(feats, repo.future_calendar, settings.forecast.horizon)
holidays = prophet_holidays(feats, future)
print(ITEM, len(feats), "days | quality", round(quality.score, 3))

## 1. Rolling-origin backtest (4 folds x 7 days)

In [ ]:
cfg = settings.forecast
factories = {
    "seasonal_naive": (lambda: SeasonalNaiveForecaster(), ()),
    "arima": (lambda: ARIMAForecaster(seasonal_order=cfg.arima_seasonal_order,
                                       order_grid=cfg.arima_order_grid,
                                       fit_window=cfg.arima_fit_window), ()),
    "prophet": (lambda: ProphetForecaster(holidays=holidays, uncertainty_samples=100),
                PROPHET_REGRESSORS),
}
results = {name: rolling_origin_backtest(f, feats, cfg.horizon, cfg.backtest_folds,
                                         regressors=regs, model_name=name)
           for name, (f, regs) in factories.items()}
results["ensemble"] = ensemble_backtest([results["arima"], results["prophet"]])
metrics = pd.DataFrame({k: v.metrics.model_dump() for k, v in results.items()}).T
metrics.sort_values("mae")

In [ ]:
fig = go.Figure()
actual = results["arima"].predictions
fig.add_scatter(x=actual["date"], y=actual["actual"], name="actual", line_color="black")
for name, res in results.items():
    fig.add_scatter(x=res.predictions["date"], y=res.predictions["yhat"], name=name,
                    line_dash="dot")
fig.update_layout(title=f"{ITEM}: backtest predictions (4 x 7 days)", height=400)
fig.show()

## 2. ARIMA order selection

In [ ]:
arima = ARIMAForecaster(seasonal_order=cfg.arima_seasonal_order,
                        order_grid=cfg.arima_order_grid, fit_window=cfg.arima_fit_window)
arima.fit(feats)
print("selected order:", arima.selected_order, "| AIC:", round(arima.aic, 1),
      "| stable:", arima.stable)

## 3. Final 7-day forecasts vs real hold-out week

In [ ]:
act = repo.future_actuals(ITEM).head(cfg.horizon)
fc = {}
for name, (f, regs) in factories.items():
    model = f().fit(feats)
    fc[name] = model.predict(future[["date", *regs]])["yhat"].to_numpy()
fc["ensemble"] = (fc["arima"] + fc["prophet"]) / 2
holdout = pd.DataFrame({k: compute_metrics(act["units"], v).model_dump() for k, v in fc.items()}).T
holdout.sort_values("mae")

In [ ]:
fig = go.Figure()
hist = feats.tail(42)
fig.add_scatter(x=hist["date"], y=hist["units"], name="history", line_color="#999")
fig.add_scatter(x=act["date"], y=act["units"], name="actual (hold-out)", line_color="black")
for name, y in fc.items():
    fig.add_scatter(x=future["date"], y=y, name=name, line_dash="dot")
fig.update_layout(title=f"{ITEM}: 7-day forecasts vs actual", height=400)
fig.show()

## 4. The Forecast Agent end-to-end

The agent wraps the steps above, adds plausibility checks, model agreement and skill vs naive, and returns a typed `ForecastOutput`.

In [ ]:
from src.agents.forecast_agent import ForecastAgent
from src.agents.llm_client import TemplateLLMClient

agent = ForecastAgent(settings, TemplateLLMClient())
out = agent.run(ITEM, feats, future, holidays)
print("model used:", out.model_used, "| 7-day total:", round(out.total_forecast, 1),
      "| agreement:", out.model_agreement_pct, "| skill vs naive:", out.skill_vs_naive)
pd.DataFrame([p.model_dump() for p in out.forecast])

## Findings

* No single model wins everywhere - per-item selection by backtest MAE beats any fixed choice on the hold-out (see `docs/evaluation_report.md`).
* The ensemble is the safest fallback when ARIMA and Prophet disagree, which is exactly what the Critic requests on revision.
* Unconstrained ARIMA can explode on intermittent series; the sanity checks and fallback orders in `ARIMAForecaster` are required in production.